# Libreria SDV
*Autor: Alex Bolaños Sánchez 
*Fecha: 04 de junio

In [13]:
# Importamos librerias
import pandas as pd
import numpy as np
import random

In [21]:
import sdv #Generar datos sinteticos
import sdmetrics #Evaluar la calidad de los datos generados

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [15]:
from sdmetrics.reports.single_table import QualityReport

In [16]:
print("SDV", sdv.__version__)

SDV 1.37.0


In [17]:
#Creamos un DataSet (original) que servirá como fuente para crear los datos sintéticos 
dfClientes = pd.DataFrame(
    {
        "cliente_id" : [1,2,3,4,5,6,7,8,9,10],
        "edad" : [23,33,43,28,53,56,43,56,65,40],
        "ingreso_mensual" : [25000,15000,20000,10000,5000,17000,30000,12000,35000,7500],
        "ciudad" : ["veracruz","Córdoba","Paso del Macho","Amatlan","Fortín","Cuitláhuac","Yanga","Córdoba","Orizaba","Cuitláhuac"]
    }
)

In [18]:
dfClientes.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000,veracruz
1,2,33,15000,Córdoba
2,3,43,20000,Paso del Macho
3,4,28,10000,Amatlan
4,5,53,5000,Fortín


In [23]:
metadata = SingleTableMetadata()

In [24]:
metadata.detect_from_dataframe(
    data = dfClientes
)

In [25]:
##instalar graphviz
metadata.to_dict()

{'primary_key': 'cliente_id',
 'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}},
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}

In [26]:
#Guardamos el metadata en un archivo JSON
metadata.save_to_json(
    "clientes_metadata.json"
)

ValueError: A file named 'clientes_metadata.json' already exists in this folder. Please specify a different filename.

In [27]:
#Entrenamos el modelo para generar los datos sinteticos 
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

C:\Users\alexb\anaconda3\Lib\site-packages\sdv\single_table\base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\alexb\anaconda3\Lib\site-packages\sdv\single_table\base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [28]:
#Entrenamiento 
synthesizer.fit(
    dfClientes
)

In [31]:
#Generamos los datos sintéticos
clientes_sinteticos = synthesizer.sample(
    num_rows = 100
)

In [32]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,14436950,44,30292,Yanga
1,4022082,35,26999,Yanga
2,9005540,39,24496,Amatlan
3,3149964,40,25135,Paso del Macho
4,7635119,36,27375,Paso del Macho


In [33]:
clientes_sinteticos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   cliente_id       100 non-null    int64 
 1   edad             100 non-null    int64 
 2   ingreso_mensual  100 non-null    int64 
 3   ciudad           100 non-null    object
dtypes: int64(3), object(1)
memory usage: 3.3+ KB


In [34]:
clientes_sinteticos.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Córdoba
freq,NaN,NaN,NaN,22
mean,8.999604e+06,43.040000,23329.340000,NaN
std,4.739210e+06,11.293084,8735.883301,NaN
min,4.733400e+04,28.000000,6334.000000,NaN
25%,5.186292e+06,33.000000,16940.750000,NaN
50%,9.247919e+06,41.000000,25106.500000,NaN
75%,1.318596e+07,51.000000,31330.250000,NaN


In [35]:
#DataFrame contaminado 
dfClientesGIGO = clientes_sinteticos.copy()

In [37]:
# Colocar edades imposibles
indices = random.sample(list(dfClientesGIGO.index),5)

In [38]:
dfClientesGIGO.loc[indices, "edad"] = -5

In [39]:
# Introducimos valores duplicados
duplicados = dfClientesGIGO.sample(10, random_state = 42)

In [41]:
dfClientesGIGO = pd.concat([dfClientesGIGO, duplicados], ignore_index = True)

In [42]:
#Verificamos valores nulos 
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [44]:
dfClientesGIGO.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Córdoba
freq,NaN,NaN,NaN,23
mean,8.994359e+06,41.400000,23359.554545,NaN
std,4.747059e+06,15.151347,8896.253567,NaN
min,4.733400e+04,-5.000000,6334.000000,NaN
25%,5.223562e+06,32.250000,16792.250000,NaN
50%,9.247919e+06,41.500000,25106.500000,NaN
75%,1.315305e+07,50.500000,31466.750000,NaN


In [46]:
#Generamos el reporte
reporte = QualityReport()

In [48]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 213.89it/s]|
Column Shapes Score: 79.33%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 112.75it/s]|
Column Pair Trends Score: 12.0%

Overall Score (Average): 45.67%

